# 01b Ingest OpenSky Live

Ingest optional near-real-time OpenSky REST snapshots for the Frankfurt scope into the Bronze live table using OAuth2 client credentials.


## Run Modes

This notebook supports three execution modes:

1. `once` for a single latest safe snapshot
2. `loop` for development or demo collection on a warm cluster
3. `catch_up` for scheduled recovery using expected snapshots minus successful snapshots

The workflow writes to:

- `adsb_airspace_eddf.brz_adsb.live_states`
- `adsb_airspace_eddf.obs.ingestion_log`
- `adsb_airspace_eddf.obs.ingestion_partition_log`
- `adsb_airspace_eddf.obs.live_snapshot_manifest`


In [ ]:
from __future__ import annotations

from datetime import date, datetime, timedelta, timezone
from hashlib import sha1
from pathlib import Path
import json
import sys
import time as time_module
import uuid
import yaml

from pyspark.sql import Window
from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

UTC = timezone.utc
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.ingestion.opensky_live_client import (
    OpenSkyAPIError,
    OpenSkyLiveClient,
    OpenSkyOAuth2TokenManager,
    OpenSkyRateLimitError,
    build_config,
    normalize_states_payload,
)

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)
with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def coalesce_text(*values) -> str:
    for value in values:
        if value is None:
            continue
        text = str(value).strip()
        if text:
            return text
    return ""

def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() == "true"

def parse_int(raw_value: str) -> int:
    return int(raw_value.strip())

def parse_float(raw_value: str) -> float:
    return float(raw_value.strip())

def utc_now() -> datetime:
    return datetime.now(UTC)

def naive_utc(value: datetime | None) -> datetime | None:
    if value is None:
        return None
    if value.tzinfo is None:
        return value
    return value.astimezone(UTC).replace(tzinfo=None)

def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    return value

def escape_sql_literal(value: str) -> str:
    return value.replace("'", "''")

def floor_to_interval(epoch_seconds: int, interval_seconds: int) -> int:
    return epoch_seconds - (epoch_seconds % interval_seconds)

def build_scope_id(*, focus_airport: str, bbox_tuple: tuple[float, float, float, float], poll_interval_seconds: int) -> str:
    canonical = "|".join([
        focus_airport,
        *(f"{value:.6f}" for value in bbox_tuple),
        str(poll_interval_seconds),
    ])
    bbox_hash = sha1(canonical.encode("utf-8")).hexdigest()[:12]
    return f"{focus_airport.lower()}_live_bbox_{bbox_hash}_poll{poll_interval_seconds}_v1"

def try_get_secret(scope: str, key: str) -> str:
    if "dbutils" not in globals():
        return ""
    if not scope or not key:
        return ""
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception:
        return ""

def ensure_target_table_exists(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Missing target table: {table_name}. Run 00_platform_setup_catalog_schema.ipynb first.")

def ensure_live_snapshot_manifest_table(table_name: str) -> None:
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            scope_id STRING NOT NULL,
            snapshot_time TIMESTAMP NOT NULL,
            snapshot_epoch BIGINT NOT NULL,
            source_object STRING NOT NULL,
            target_table STRING NOT NULL,
            status STRING NOT NULL,
            rows_read BIGINT,
            rows_written BIGINT,
            requested_at TIMESTAMP NOT NULL,
            completed_at TIMESTAMP,
            run_id STRING NOT NULL,
            error_message STRING
        ) USING DELTA
        COMMENT 'Manifest of live snapshot ingestion attempts and completion status.'
        """
    )

def create_dataframe_for_table(*, target_table: str, rows: list[dict]):
    target_schema = spark.table(target_table).schema
    normalized_rows = [
        {field.name: normalize_scalar(row.get(field.name)) for field in target_schema}
        for row in rows
    ]
    return spark.createDataFrame(normalized_rows, schema=target_schema)

def append_log_row(*, target_table: str, payload: dict) -> None:
    log_df = create_dataframe_for_table(target_table=target_table, rows=[payload])
    log_df.write.mode("append").saveAsTable(target_table)

def write_manifest_row(*, target_table: str, payload: dict) -> None:
    manifest_df = create_dataframe_for_table(target_table=target_table, rows=[payload])
    manifest_df.write.mode("append").saveAsTable(target_table)

def load_successful_snapshot_epochs(*, target_table: str, scope_id: str, start_epoch: int, end_epoch: int) -> set[int]:
    scope_literal = escape_sql_literal(scope_id)
    rows = spark.sql(
        f"""
        SELECT DISTINCT snapshot_epoch
        FROM {target_table}
        WHERE scope_id = '{scope_literal}'
          AND status = 'success'
          AND snapshot_epoch BETWEEN {int(start_epoch)} AND {int(end_epoch)}
        """
    ).collect()
    return {int(row["snapshot_epoch"]) for row in rows}

def load_latest_snapshot_status(*, target_table: str, scope_id: str, snapshot_epoch: int):
    scope_literal = escape_sql_literal(scope_id)
    row = spark.sql(
        f"""
        SELECT status, rows_written, completed_at
        FROM {target_table}
        WHERE scope_id = '{scope_literal}'
          AND snapshot_epoch = {int(snapshot_epoch)}
        ORDER BY requested_at DESC, completed_at DESC
        LIMIT 1
        """
    ).first()
    return row

def load_oldest_unresolved_snapshot_epoch(*, target_table: str, scope_id: str):
    scope_literal = escape_sql_literal(scope_id)
    unresolved_df = spark.sql(
        f"""
        SELECT snapshot_epoch, status
        FROM (
            SELECT
                snapshot_epoch,
                status,
                ROW_NUMBER() OVER (
                    PARTITION BY snapshot_epoch
                    ORDER BY requested_at DESC, completed_at DESC
                ) AS row_number
            FROM {target_table}
            WHERE scope_id = '{scope_literal}'
        ) latest
        WHERE row_number = 1
          AND status <> 'success'
        ORDER BY snapshot_epoch ASC
        LIMIT 1
        """
    )
    row = unresolved_df.first()
    return None if row is None else int(row["snapshot_epoch"])

def merge_rows_into_live_bronze(*, target_table: str, rows: list[dict]) -> int:
    if not rows:
        return 0
    stage_view = f"live_snapshot_stage_{uuid.uuid4().hex[:8]}"
    spark_df = create_dataframe_for_table(target_table=target_table, rows=rows)
    spark_df.createOrReplaceTempView(stage_view)
    column_names = spark.table(target_table).columns
    assignments = ",\n                ".join([f"target.{column_name} = source.{column_name}" for column_name in column_names])
    inserts = ", ".join(column_names)
    insert_values = ", ".join([f"source.{column_name}" for column_name in column_names])
    try:
        spark.sql(
            f"""
            MERGE INTO {target_table} AS target
            USING {stage_view} AS source
            ON target.snapshot_time = source.snapshot_time
            AND target.icao24 = source.icao24
            WHEN MATCHED THEN UPDATE SET
                {assignments}
            WHEN NOT MATCHED THEN INSERT ({inserts}) VALUES ({insert_values})
            """
        )
    finally:
        spark.catalog.dropTempView(stage_view)
    return len(rows)

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
live_connection_defaults = pipeline_config.get("opensky_live_connection", {})
live_ingestion_defaults = pipeline_config.get("live_ingestion", {})
bbox_defaults = region_config["regional_bbox"]

default_mode = str(live_ingestion_defaults.get("default_mode", "loop"))
default_poll_interval_seconds = str(live_ingestion_defaults.get("poll_interval_seconds", 60))
default_run_duration_minutes = str(live_ingestion_defaults.get("run_duration_minutes", 15))
default_run_budget_seconds = str(live_ingestion_defaults.get("run_budget_seconds", 720))
default_safety_buffer_seconds = str(live_ingestion_defaults.get("safety_buffer_seconds", 90))
default_recoverable_lookback_seconds = str(live_ingestion_defaults.get("recoverable_lookback_seconds", 3300))
default_max_snapshots_per_run = str(live_ingestion_defaults.get("max_snapshots_per_run", 80))
default_live_base_url = str(live_connection_defaults.get("base_url", "https://opensky-network.org/api"))
default_live_token_url = str(live_connection_defaults.get("token_url", "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"))
default_live_secret_scope = str(live_connection_defaults.get("secret_scope", "opensky"))
default_live_client_id_key = str(live_connection_defaults.get("client_id_key", "live_client_id"))
default_live_client_secret_key = str(live_connection_defaults.get("client_secret_key", "live_client_secret"))

catalog = default_catalog
mode_raw = default_mode
poll_interval_seconds_raw = default_poll_interval_seconds
run_duration_minutes_raw = default_run_duration_minutes
run_budget_seconds_raw = default_run_budget_seconds
safety_buffer_seconds_raw = default_safety_buffer_seconds
recoverable_lookback_seconds_raw = default_recoverable_lookback_seconds
max_snapshots_per_run_raw = default_max_snapshots_per_run
bbox_min_lat_raw = str(bbox_defaults["min_latitude"])
bbox_max_lat_raw = str(bbox_defaults["max_latitude"])
bbox_min_lon_raw = str(bbox_defaults["min_longitude"])
bbox_max_lon_raw = str(bbox_defaults["max_longitude"])
dry_run_raw = "true"
live_base_url_raw = default_live_base_url
live_token_url_raw = default_live_token_url
live_secret_scope_raw = default_live_secret_scope
live_client_id_key_raw = default_live_client_id_key
live_client_secret_key_raw = default_live_client_secret_key

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.dropdown("mode", default_mode, ["once", "loop", "catch_up"])
    dbutils.widgets.text("poll_interval_seconds", default_poll_interval_seconds)
    dbutils.widgets.text("run_duration_minutes", default_run_duration_minutes)
    dbutils.widgets.text("run_budget_seconds", default_run_budget_seconds)
    dbutils.widgets.text("safety_buffer_seconds", default_safety_buffer_seconds)
    dbutils.widgets.text("recoverable_lookback_seconds", default_recoverable_lookback_seconds)
    dbutils.widgets.text("max_snapshots_per_run", default_max_snapshots_per_run)
    dbutils.widgets.text("bbox_min_latitude", str(bbox_defaults["min_latitude"]))
    dbutils.widgets.text("bbox_max_latitude", str(bbox_defaults["max_latitude"]))
    dbutils.widgets.text("bbox_min_longitude", str(bbox_defaults["min_longitude"]))
    dbutils.widgets.text("bbox_max_longitude", str(bbox_defaults["max_longitude"]))
    dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])
    dbutils.widgets.text("live_base_url", default_live_base_url)
    dbutils.widgets.text("live_token_url", default_live_token_url)
    dbutils.widgets.text("live_secret_scope", default_live_secret_scope)
    dbutils.widgets.text("live_client_id_key", default_live_client_id_key)
    dbutils.widgets.text("live_client_secret_key", default_live_client_secret_key)

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    mode_raw = dbutils.widgets.get("mode").strip() or default_mode
    poll_interval_seconds_raw = dbutils.widgets.get("poll_interval_seconds").strip() or default_poll_interval_seconds
    run_duration_minutes_raw = dbutils.widgets.get("run_duration_minutes").strip() or default_run_duration_minutes
    run_budget_seconds_raw = dbutils.widgets.get("run_budget_seconds").strip() or default_run_budget_seconds
    safety_buffer_seconds_raw = dbutils.widgets.get("safety_buffer_seconds").strip() or default_safety_buffer_seconds
    recoverable_lookback_seconds_raw = dbutils.widgets.get("recoverable_lookback_seconds").strip() or default_recoverable_lookback_seconds
    max_snapshots_per_run_raw = dbutils.widgets.get("max_snapshots_per_run").strip() or default_max_snapshots_per_run
    bbox_min_lat_raw = dbutils.widgets.get("bbox_min_latitude").strip() or str(bbox_defaults["min_latitude"])
    bbox_max_lat_raw = dbutils.widgets.get("bbox_max_latitude").strip() or str(bbox_defaults["max_latitude"])
    bbox_min_lon_raw = dbutils.widgets.get("bbox_min_longitude").strip() or str(bbox_defaults["min_longitude"])
    bbox_max_lon_raw = dbutils.widgets.get("bbox_max_longitude").strip() or str(bbox_defaults["max_longitude"])
    dry_run_raw = dbutils.widgets.get("dry_run")
    live_base_url_raw = dbutils.widgets.get("live_base_url").strip() or default_live_base_url
    live_token_url_raw = dbutils.widgets.get("live_token_url").strip() or default_live_token_url
    live_secret_scope_raw = dbutils.widgets.get("live_secret_scope").strip() or default_live_secret_scope
    live_client_id_key_raw = dbutils.widgets.get("live_client_id_key").strip() or default_live_client_id_key
    live_client_secret_key_raw = dbutils.widgets.get("live_client_secret_key").strip() or default_live_client_secret_key

mode = mode_raw.strip().lower()
if mode not in {"once", "loop", "catch_up"}:
    raise ValueError("mode must be one of once, loop, catch_up")

poll_interval_seconds = parse_int(poll_interval_seconds_raw)
run_duration_minutes = parse_int(run_duration_minutes_raw)
run_budget_seconds = parse_int(run_budget_seconds_raw)
safety_buffer_seconds = parse_int(safety_buffer_seconds_raw)
recoverable_lookback_seconds = parse_int(recoverable_lookback_seconds_raw)
max_snapshots_per_run = parse_int(max_snapshots_per_run_raw)
dry_run = parse_bool(dry_run_raw)

bbox_tuple = (
    parse_float(bbox_min_lat_raw),
    parse_float(bbox_max_lat_raw),
    parse_float(bbox_min_lon_raw),
    parse_float(bbox_max_lon_raw),
)
if bbox_tuple[0] >= bbox_tuple[1] or bbox_tuple[2] >= bbox_tuple[3]:
    raise ValueError("Bounding box must satisfy min < max for latitude and longitude")
if poll_interval_seconds <= 0 or run_duration_minutes <= 0 or run_budget_seconds <= 0:
    raise ValueError("Timing parameters must be positive")
if max_snapshots_per_run <= 0 or safety_buffer_seconds < 0 or recoverable_lookback_seconds <= 0:
    raise ValueError("Live ingestion limits must be positive and safety_buffer_seconds must be non-negative")

focus_airport = region_config["focus_airport"]
ingestion_radius_nm = int(region_config.get("ingestion_radius_nm", 0))
scope_id = build_scope_id(
    focus_airport=focus_airport,
    bbox_tuple=bbox_tuple,
    poll_interval_seconds=poll_interval_seconds,
)
live_table = f"{catalog}.brz_adsb.live_states"
ingestion_log_table = f"{catalog}.obs.ingestion_log"
partition_log_table = f"{catalog}.obs.ingestion_partition_log"
manifest_table = f"{catalog}.obs.live_snapshot_manifest"

ensure_target_table_exists(live_table)
ensure_target_table_exists(ingestion_log_table)
ensure_target_table_exists(partition_log_table)
ensure_live_snapshot_manifest_table(manifest_table)

run_started_at = utc_now()
latest_safe_epoch = floor_to_interval(int(run_started_at.timestamp()) - safety_buffer_seconds, poll_interval_seconds)
first_expected_epoch = floor_to_interval(latest_safe_epoch - recoverable_lookback_seconds, poll_interval_seconds)
successful_epochs = load_successful_snapshot_epochs(
    target_table=manifest_table,
    scope_id=scope_id,
    start_epoch=first_expected_epoch,
    end_epoch=latest_safe_epoch,
)
expected_epochs = list(range(first_expected_epoch, latest_safe_epoch + 1, poll_interval_seconds))
pending_epochs = [epoch for epoch in expected_epochs if epoch not in successful_epochs]

if mode == "once":
    planned_snapshot_epochs = [latest_safe_epoch]
elif mode == "loop":
    planned_loop_iterations = max(1, (run_duration_minutes * 60 + poll_interval_seconds - 1) // poll_interval_seconds)
    planned_snapshot_epochs = [latest_safe_epoch]
else:
    planned_loop_iterations = 0
    planned_snapshot_epochs = pending_epochs[:max_snapshots_per_run]

run_id = f"live_ingest_{utc_now().strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
run_plan = {
    "status": "dry_run" if dry_run else "planned",
    "run_id": run_id,
    "mode": mode,
    "scope_id": scope_id,
    "focus_airport": focus_airport,
    "bbox": {
        "min_latitude": bbox_tuple[0],
        "max_latitude": bbox_tuple[1],
        "min_longitude": bbox_tuple[2],
        "max_longitude": bbox_tuple[3],
    },
    "poll_interval_seconds": poll_interval_seconds,
    "run_duration_minutes": run_duration_minutes,
    "run_budget_seconds": run_budget_seconds,
    "safety_buffer_seconds": safety_buffer_seconds,
    "recoverable_lookback_seconds": recoverable_lookback_seconds,
    "max_snapshots_per_run": max_snapshots_per_run,
    "latest_safe_epoch": latest_safe_epoch,
    "first_expected_epoch": first_expected_epoch,
    "pending_snapshot_count": len(pending_epochs),
    "planned_snapshot_count": len(planned_snapshot_epochs) if mode != "loop" else planned_loop_iterations,
    "target_tables": {
        "live": live_table,
        "run_log": ingestion_log_table,
        "partition_log": partition_log_table,
        "manifest": manifest_table,
    },
    "oauth2": {
        "base_url": live_base_url_raw,
        "token_url": live_token_url_raw,
        "secret_scope": live_secret_scope_raw,
        "client_id_key": live_client_id_key_raw,
        "client_secret_key": live_client_secret_key_raw,
    },
}

run_plan


In [ ]:
def ingest_snapshot_epoch(*, client: OpenSkyLiveClient, snapshot_epoch: int) -> dict:
    partition_started_at = utc_now()
    existing_status_row = load_latest_snapshot_status(
        target_table=manifest_table,
        scope_id=scope_id,
        snapshot_epoch=snapshot_epoch,
    )
    if existing_status_row is not None and existing_status_row["status"] == "success":
        append_log_row(
            target_table=partition_log_table,
            payload={
                "run_id": run_id,
                "pipeline_name": "01b_ingest_opensky_live",
                "source_object": "states/all",
                "target_table": live_table,
                "partition_key": "snapshot_epoch",
                "partition_value": str(snapshot_epoch),
                "status": "duplicate_snapshot_complete",
                "rows_read": 0,
                "rows_written": int(existing_status_row["rows_written"] or 0),
                "attempt_no": 1,
                "dry_run": False,
                "started_at": naive_utc(partition_started_at),
                "completed_at": naive_utc(utc_now()),
                "error_message": None,
            },
        )
        return {
            "snapshot_epoch": snapshot_epoch,
            "actual_snapshot_epoch": snapshot_epoch,
            "status": "duplicate_snapshot_complete",
            "rows_read": 0,
            "rows_written": int(existing_status_row["rows_written"] or 0),
            "retry_after_seconds": None,
            "error_message": None,
        }

    status = "failed"
    rows_read = 0
    rows_written = 0
    retry_after_seconds = None
    error_message = None
    actual_snapshot_epoch = snapshot_epoch
    actual_snapshot_time = datetime.fromtimestamp(snapshot_epoch, tz=UTC).replace(tzinfo=None)
    try:
        payload = client.fetch_states_all(time=snapshot_epoch, bbox=bbox_tuple)
        actual_snapshot_epoch = int(payload["time"])
        actual_snapshot_time = datetime.fromtimestamp(actual_snapshot_epoch, tz=UTC).replace(tzinfo=None)
        rows = normalize_states_payload(
            payload,
            run_id=run_id,
            ingested_at=utc_now(),
            scope_airport=focus_airport,
            scope_radius_nm=ingestion_radius_nm,
        )
        rows_read = len(rows)
        rows_written = merge_rows_into_live_bronze(target_table=live_table, rows=rows)
        status = "success"
        write_manifest_row(
            target_table=manifest_table,
            payload={
                "scope_id": scope_id,
                "snapshot_time": actual_snapshot_time,
                "snapshot_epoch": actual_snapshot_epoch,
                "source_object": "states/all",
                "target_table": live_table,
                "status": status,
                "rows_read": rows_read,
                "rows_written": rows_written,
                "requested_at": naive_utc(partition_started_at),
                "completed_at": naive_utc(utc_now()),
                "run_id": run_id,
                "error_message": None,
            },
        )
    except OpenSkyRateLimitError as exc:
        status = "rate_limited"
        retry_after_seconds = exc.retry_after_seconds
        error_message = str(exc)
        write_manifest_row(
            target_table=manifest_table,
            payload={
                "scope_id": scope_id,
                "snapshot_time": actual_snapshot_time,
                "snapshot_epoch": actual_snapshot_epoch,
                "source_object": "states/all",
                "target_table": live_table,
                "status": status,
                "rows_read": rows_read,
                "rows_written": rows_written,
                "requested_at": naive_utc(partition_started_at),
                "completed_at": naive_utc(utc_now()),
                "run_id": run_id,
                "error_message": error_message,
            },
        )
    except OpenSkyAPIError as exc:
        error_message = str(exc)
        write_manifest_row(
            target_table=manifest_table,
            payload={
                "scope_id": scope_id,
                "snapshot_time": actual_snapshot_time,
                "snapshot_epoch": actual_snapshot_epoch,
                "source_object": "states/all",
                "target_table": live_table,
                "status": status,
                "rows_read": rows_read,
                "rows_written": rows_written,
                "requested_at": naive_utc(partition_started_at),
                "completed_at": naive_utc(utc_now()),
                "run_id": run_id,
                "error_message": error_message,
            },
        )
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        write_manifest_row(
            target_table=manifest_table,
            payload={
                "scope_id": scope_id,
                "snapshot_time": actual_snapshot_time,
                "snapshot_epoch": actual_snapshot_epoch,
                "source_object": "states/all",
                "target_table": live_table,
                "status": status,
                "rows_read": rows_read,
                "rows_written": rows_written,
                "requested_at": naive_utc(partition_started_at),
                "completed_at": naive_utc(utc_now()),
                "run_id": run_id,
                "error_message": error_message,
            },
        )
    finally:
        append_log_row(
            target_table=partition_log_table,
            payload={
                "run_id": run_id,
                "pipeline_name": "01b_ingest_opensky_live",
                "source_object": "states/all",
                "target_table": live_table,
                "partition_key": "snapshot_epoch",
                "partition_value": str(snapshot_epoch),
                "status": status,
                "rows_read": rows_read,
                "rows_written": rows_written,
                "attempt_no": 1,
                "dry_run": False,
                "started_at": naive_utc(partition_started_at),
                "completed_at": naive_utc(utc_now()),
                "error_message": error_message,
            },
        )

    return {
        "snapshot_epoch": snapshot_epoch,
        "actual_snapshot_epoch": actual_snapshot_epoch,
        "status": status,
        "rows_read": rows_read,
        "rows_written": rows_written,
        "retry_after_seconds": retry_after_seconds,
        "error_message": error_message,
    }

def build_catch_up_queue(now_utc: datetime) -> dict:
    latest_epoch = floor_to_interval(int(now_utc.timestamp()) - safety_buffer_seconds, poll_interval_seconds)
    first_epoch = floor_to_interval(latest_epoch - recoverable_lookback_seconds, poll_interval_seconds)
    success_epochs = load_successful_snapshot_epochs(
        target_table=manifest_table,
        scope_id=scope_id,
        start_epoch=first_epoch,
        end_epoch=latest_epoch,
    )
    expected = list(range(first_epoch, latest_epoch + 1, poll_interval_seconds))
    pending = [epoch for epoch in expected if epoch not in success_epochs]
    return {
        "latest_safe_epoch": latest_epoch,
        "first_expected_epoch": first_epoch,
        "expected_epochs": expected,
        "pending_epochs": pending,
    }

def maybe_sleep_for_rate_limit(*, retry_after_seconds: int | None, deadline: datetime | None = None) -> None:
    if retry_after_seconds is None or retry_after_seconds <= 0:
        return
    if deadline is None:
        time_module.sleep(retry_after_seconds)
        return
    remaining_seconds = max(0, int((deadline - utc_now()).total_seconds()))
    if remaining_seconds <= 0:
        return
    time_module.sleep(min(retry_after_seconds, remaining_seconds))

if dry_run:
    dry_run_summary = run_plan
    dry_run_summary
else:
    live_client_id = try_get_secret(live_secret_scope_raw, live_client_id_key_raw)
    live_client_secret = try_get_secret(live_secret_scope_raw, live_client_secret_key_raw)
    live_config = build_config(
        base_url=live_base_url_raw,
        token_url=live_token_url_raw,
        client_id=live_client_id,
        client_secret=live_client_secret,
        request_timeout_seconds=30,
    )
    client = OpenSkyLiveClient(live_config, token_manager=OpenSkyOAuth2TokenManager(live_config))

    result_summaries = []
    run_deadline = run_started_at + timedelta(seconds=run_budget_seconds)

    if mode == "once":
        result_summaries.append(ingest_snapshot_epoch(client=client, snapshot_epoch=latest_safe_epoch))
    elif mode == "loop":
        loop_deadline = run_started_at + timedelta(minutes=run_duration_minutes)
        while utc_now() < loop_deadline:
            current_snapshot_epoch = floor_to_interval(int(utc_now().timestamp()) - safety_buffer_seconds, poll_interval_seconds)
            attempt_summary = ingest_snapshot_epoch(client=client, snapshot_epoch=current_snapshot_epoch)
            result_summaries.append(attempt_summary)
            maybe_sleep_for_rate_limit(
                retry_after_seconds=attempt_summary["retry_after_seconds"],
                deadline=loop_deadline,
            )
            next_epoch = current_snapshot_epoch + poll_interval_seconds
            target_wake_epoch = next_epoch + safety_buffer_seconds
            sleep_seconds = max(0, target_wake_epoch - int(utc_now().timestamp()))
            if sleep_seconds <= 0:
                continue
            remaining = max(0, int((loop_deadline - utc_now()).total_seconds()))
            if remaining <= 0:
                break
            time_module.sleep(min(sleep_seconds, remaining))
    else:
        catch_up_plan = build_catch_up_queue(run_started_at)
        oldest_unresolved_epoch = load_oldest_unresolved_snapshot_epoch(
            target_table=manifest_table,
            scope_id=scope_id,
        )
        if oldest_unresolved_epoch is not None and oldest_unresolved_epoch < catch_up_plan["first_expected_epoch"]:
            raise RuntimeError("Live backlog exceeded recoverable REST lookback window.")
        for snapshot_epoch in catch_up_plan["pending_epochs"][:max_snapshots_per_run]:
            if utc_now() >= run_deadline:
                break
            attempt_summary = ingest_snapshot_epoch(client=client, snapshot_epoch=snapshot_epoch)
            result_summaries.append(attempt_summary)
            maybe_sleep_for_rate_limit(
                retry_after_seconds=attempt_summary["retry_after_seconds"],
                deadline=run_deadline,
            )

    success_partition_count = sum(1 for item in result_summaries if item["status"] == "success")
    failed_partition_count = sum(1 for item in result_summaries if item["status"] in {"failed", "rate_limited"})
    duplicate_partition_count = sum(1 for item in result_summaries if item["status"] == "duplicate_snapshot_complete")
    rows_written_total = sum(int(item["rows_written"] or 0) for item in result_summaries)
    if failed_partition_count == 0:
        final_status = "success"
    elif success_partition_count > 0 or duplicate_partition_count > 0:
        final_status = "partial_success"
    else:
        final_status = "failed"

    append_log_row(
        target_table=ingestion_log_table,
        payload={
            "run_id": run_id,
            "pipeline_name": "01b_ingest_opensky_live",
            "source_name": "opensky_rest",
            "source_object": "states/all",
            "target_table": live_table,
            "scope_id": scope_id,
            "start_date": datetime.fromtimestamp(first_expected_epoch, tz=UTC).date() if mode == "catch_up" else datetime.fromtimestamp(latest_safe_epoch, tz=UTC).date(),
            "end_date": datetime.fromtimestamp(latest_safe_epoch, tz=UTC).date(),
            "partition_key": "snapshot_epoch",
            "planned_partition_count": run_plan["planned_snapshot_count"],
            "success_partition_count": success_partition_count,
            "failed_partition_count": failed_partition_count,
            "dry_run": False,
            "status": final_status,
            "rows_written": rows_written_total,
            "started_at": naive_utc(run_started_at),
            "completed_at": naive_utc(utc_now()),
            "error_message": None if failed_partition_count == 0 else "One or more live snapshot attempts failed or were rate-limited.",
            "metadata_json": json.dumps(
                {
                    "mode": mode,
                    "bbox": {
                        "min_latitude": bbox_tuple[0],
                        "max_latitude": bbox_tuple[1],
                        "min_longitude": bbox_tuple[2],
                        "max_longitude": bbox_tuple[3],
                    },
                    "poll_interval_seconds": poll_interval_seconds,
                    "run_budget_seconds": run_budget_seconds,
                    "run_duration_minutes": run_duration_minutes,
                    "safety_buffer_seconds": safety_buffer_seconds,
                    "recoverable_lookback_seconds": recoverable_lookback_seconds,
                    "max_snapshots_per_run": max_snapshots_per_run,
                    "duplicate_partition_count": duplicate_partition_count,
                },
                sort_keys=True,
            ),
        },
    )

    final_summary = {
        "status": final_status,
        "run_id": run_id,
        "mode": mode,
        "scope_id": scope_id,
        "latest_safe_epoch": latest_safe_epoch,
        "planned_snapshot_count": run_plan["planned_snapshot_count"],
        "success_partition_count": success_partition_count,
        "duplicate_partition_count": duplicate_partition_count,
        "failed_partition_count": failed_partition_count,
        "rows_written": rows_written_total,
    }

    final_summary
